[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ThanhDatVN/AI_journey/blob/main/notebooks/09_naive_bayes/03_multinomial_bernoulli_text.ipynb)

# Bài 03 — Multinomial & Bernoulli Naive Bayes cho văn bản

**🎯 Mục tiêu**
- Biểu diễn văn bản bằng **bag-of-words** (`CountVectorizer`).
- Cài đặt **Multinomial NB** from scratch (Laplace smoothing) — baseline kinh điển cho phân loại văn bản.
- Phân biệt Multinomial (đếm số lần) vs **Bernoulli** (có/không, phạt cả từ vắng mặt) vs **Complement NB**.
- Đọc model: từ nào "đẩy" về lớp nào; thấy giới hạn của bag-of-words với **phủ định**.

In [ ]:
# ⚙️ Setup — chạy được cả local lẫn Google Colab (local: đã `pip install -e .` thì bỏ qua)
import sys
if "google.colab" in sys.modules:
    !test -d /content/ml-course || git clone -q --depth 1 https://github.com/ThanhDatVN/AI_journey /content/ml-course
    !pip install -q -e /content/ml-course
    sys.path.insert(0, "/content/ml-course/src")
    %cd /content/ml-course/notebooks/09_naive_bayes

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.model_selection import cross_val_score, train_test_split
from sklearn.naive_bayes import BernoulliNB, ComplementNB, MultinomialNB
from sklearn.pipeline import make_pipeline

from ml_course.visualization import set_style

set_style()
reviews = pd.read_csv("../../data/sample/reviews_vi.csv")
print(reviews.shape, "| tỉ lệ tích cực:", reviews.label.mean().round(3))
reviews.head()

## 1. Bag-of-words

Mỗi văn bản → vector đếm số lần mỗi từ (hoặc cụm từ — n-gram) xuất hiện; **bỏ qua thứ tự**. Kết quả là ma trận **thưa** (*sparse*) kích thước (số văn bản × kích thước từ điển).

In [ ]:
docs = ["pin trâu, rất đáng tiền", "pin yếu, không đáng tiền", "rất thích, rất đáng tiền"]
cv = CountVectorizer()
counts = cv.fit_transform(docs)
pd.DataFrame(counts.toarray(), columns=cv.get_feature_names_out(), index=docs)

## 2. Multinomial Naive Bayes

Coi văn bản là dãy từ rút ngẫu nhiên từ một "túi từ" riêng của mỗi lớp với xác suất $\theta_{cw} = P(w \mid y = c)$:

$$\log P(y = c \mid \mathbf x) = \log P(y = c) + \sum_{w} x_w \log\theta_{cw} + \text{const}, \qquad \hat\theta_{cw} = \frac{N_{cw} + \alpha}{\sum_{w'} N_{cw'} + \alpha |V|}$$

trong đó $N_{cw}$ = tổng số lần từ $w$ xuất hiện trong các văn bản lớp $c$, $|V|$ = kích thước từ điển. Chú ý: đây là một **model tuyến tính** theo $\mathbf x$ — giống Logistic Regression nhưng trọng số được **đếm** chứ không phải tối ưu.

In [ ]:
class MultinomialNaiveBayes:
    def __init__(self, alpha=1.0):
        self.alpha = alpha

    def fit(self, X, y):
        y = np.asarray(y)
        self.classes_ = np.unique(y)
        counts = np.vstack([np.asarray(X[y == c].sum(axis=0)).ravel() for c in self.classes_]) + self.alpha  # (C, V)
        self.log_theta_ = np.log(counts / counts.sum(axis=1, keepdims=True))
        self.log_prior_ = np.log([np.mean(y == c) for c in self.classes_])
        return self

    def joint_log_likelihood(self, X):
        return np.asarray(X @ self.log_theta_.T) + self.log_prior_          # X thưa @ dày → nhanh

    def predict(self, X):
        return self.classes_[np.argmax(self.joint_log_likelihood(X), axis=1)]


texts_train, texts_test, y_train, y_test = train_test_split(reviews.text, reviews.label, test_size=0.25,
                                                            random_state=0, stratify=reviews.label)
vec = CountVectorizer(ngram_range=(1, 2))
X_train = vec.fit_transform(texts_train)                 # fit từ điển CHỈ trên tập train
X_test = vec.transform(texts_test)
print("kích thước từ điển:", X_train.shape[1], "| tỉ lệ phần tử khác 0:", f"{X_train.nnz / np.prod(X_train.shape):.4f}")

ours = MultinomialNaiveBayes(alpha=1.0).fit(X_train, y_train.to_numpy())
sk = MultinomialNB(alpha=1.0).fit(X_train, y_train)
print(f"accuracy scratch = {np.mean(ours.predict(X_test) == y_test):.4f} | sklearn = {sk.score(X_test, y_test):.4f}")
print("tham số trùng sklearn:", np.allclose(ours.log_theta_, sk.feature_log_prob_))

> 💡 Dữ liệu này có ~3% nhãn bị lật ngẫu nhiên (`label_noise=0.03`) → không model nào vượt quá ~97% accuracy. Luôn hỏi: **trần** của bài toán (nhiễu nhãn, Bayes error) là bao nhiêu?

## 3. Đọc model: từ nào mang tín hiệu?

Với 2 lớp, log-odds đóng góp của mỗi lần xuất hiện từ $w$ là $\log\theta_{1w} - \log\theta_{0w}$:

In [ ]:
log_ratio = pd.Series(ours.log_theta_[1] - ours.log_theta_[0], index=vec.get_feature_names_out())
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
log_ratio.nlargest(12).sort_values().plot.barh(ax=axes[0], color="tab:green", title="Đẩy về TÍCH CỰC")
log_ratio.nsmallest(12).sort_values(ascending=False).plot.barh(ax=axes[1], color="tab:red", title="Đẩy về TIÊU CỰC")
plt.tight_layout()
plt.show()

## 4. Phủ định: giới hạn của bag-of-words

Trong dữ liệu train, từ "không" chỉ xuất hiện trong review tiêu cực ("không hài lòng", "không giống mô tả"...) → model học "không" = tín hiệu tiêu cực rất mạnh. Còn những câu **phủ định của điều xấu** — nghĩa là tích cực — thì sao?

In [ ]:
probe = ["rất đáng tiền", "không hề thất vọng", "giao hàng không chậm", "chất lượng không kém"]
for ngram in [(1, 1), (1, 2)]:
    pipe = make_pipeline(CountVectorizer(ngram_range=ngram), MultinomialNB())
    acc = cross_val_score(pipe, reviews.text, reviews.label, cv=5).mean()
    pipe.fit(texts_train, y_train)
    print(f"ngram {ngram}: accuracy CV = {acc:.4f}")
    for text, p in zip(probe, pipe.predict_proba(probe)[:, 1]):
        print(f"    P(tích cực | '{text}') = {p:.3f}")

Cả ba câu cuối đều mang nghĩa **tích cực** nhưng bị dự đoán tiêu cực gần như chắc chắn: model chỉ cộng bằng chứng của từng từ ("không" −, "thất vọng" −, "chậm" −). Bigram không cứu được vì "không hề", "không chậm", "không kém" **chưa từng xuất hiện** trong tập train — n-gram chỉ dùng được ngữ cảnh đã thấy. Hiểu ngữ cảnh thật sự cần biểu diễn khác: word embedding, Transformer (chương 31–32).

## 5. Bernoulli NB và Complement NB

| Biến thể | Feature | $P(\mathbf x \mid y)$ | Ghi chú |
|---|---|---|---|
| **Multinomial** | số lần xuất hiện | $\prod_w \theta_{cw}^{x_w}$ | mặc định cho văn bản dài; dùng được với TF-IDF |
| **Bernoulli** | có / không (0/1) | $\prod_w p_{cw}^{x_w}(1 - p_{cw})^{1 - x_w}$ | **phạt cả từ vắng mặt**; hợp văn bản ngắn |
| **Complement** | đếm | ước lượng từ **các lớp còn lại** | ổn định hơn khi mất cân bằng lớp |
| **Gaussian** | liên tục | Gaussian mỗi feature | bài 02 |
| **Categorical** | phân loại | bảng tần suất mỗi giá trị | feature phân loại (`CategoricalNB`) |

Khác biệt rõ nhất khi **ít dữ liệu**. Bài toán 3 lớp (`sentiment`: pos / neg / neu — lớp neu chỉ ~14%), train trên vài chục review, đo macro-F1:

In [ ]:
from sklearn.metrics import f1_score

pool, held_out = train_test_split(reviews, test_size=0.5, random_state=0, stratify=reviews.sentiment)
for m in [30, 60, 150]:
    sub = pool.sample(m, random_state=1)
    scores = {}
    for name, model in [("Multinomial", MultinomialNB()), ("Bernoulli", BernoulliNB()), ("Complement", ComplementNB())]:
        pipe = make_pipeline(CountVectorizer(ngram_range=(1, 2)), model).fit(sub.text, sub.sentiment)
        scores[name] = round(f1_score(held_out.sentiment, pipe.predict(held_out.text), average="macro"), 3)
    print(f"{m:4d} review train: {scores}")

Với rất ít dữ liệu, Bernoulli kém hẳn (nó phải ước lượng xác suất có/không cho **mọi** từ trong từ điển, kể cả từ vắng mặt, từ vài chục mẫu); Complement nhỉnh hơn Multinomial khi lớp nhỏ (neu) có quá ít mẫu. Khi đủ dữ liệu, cả ba gần như bằng nhau trên bộ dữ liệu đơn giản này.

## 6. Ảnh hưởng của α (smoothing)

$\alpha$ nhỏ → tin tưởng tần suất quan sát; $\alpha$ lớn → kéo mọi $\theta_{cw}$ về phân phối đều. Đo hai thứ: **macro-F1** khi chỉ có 40 review train (3 lớp), và **log-loss** (chất lượng xác suất, bài toán nhị phân, 5-fold CV):

In [ ]:
alphas = np.logspace(-3, 1.5, 10)
f1_small, logloss = [], []
for a in alphas:
    f1s = []
    for rep in range(10):
        sub = pool.sample(40, random_state=rep)
        pipe = make_pipeline(CountVectorizer(ngram_range=(1, 2)), MultinomialNB(alpha=a)).fit(sub.text, sub.sentiment)
        f1s.append(f1_score(held_out.sentiment, pipe.predict(held_out.text), average="macro"))
    f1_small.append(np.mean(f1s))
    logloss.append(-cross_val_score(make_pipeline(CountVectorizer(ngram_range=(1, 2)), MultinomialNB(alpha=a)),
                                    reviews.text, reviews.label, cv=5, scoring="neg_log_loss").mean())
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].semilogx(alphas, f1_small, "o-")
axes[0].set_title("macro-F1 (train 40 review, 3 lớp) — cao hơn là tốt")
axes[1].semilogx(alphas, logloss, "o-", color="tab:red")
axes[1].set_title("log-loss (nhị phân, CV) — thấp hơn là tốt")
for ax in axes:
    ax.set_xlabel("alpha")
plt.show()
for a, f, ll in zip(alphas, f1_small, logloss):
    print(f"alpha = {a:8.3f}: macro-F1 = {f:.3f} | log-loss = {ll:.3f}")

Hai metric kéo về hai phía:
- **Phân loại:** F1 gần như không đổi khi $\alpha \lesssim 0.3$, rồi **giảm mạnh** khi $\alpha \ge 3$ — với 40 review, mỗi lớp chỉ có vài chục lượt đếm từ, $\alpha$ lớn "lấn át" dữ liệu.
- **Xác suất:** log-loss lại **giảm** khi $\alpha$ tăng — xác suất bớt cực đoan nên ít bị phạt nặng bởi ~3% nhãn nhiễu.

→ $\alpha$ điều khiển cả độ chính xác lẫn độ "tự tin" của model. Chọn $\alpha$ bằng CV **theo đúng metric bạn cần**, và nếu cần xác suất tốt, calibrate riêng (bài 04, chương 37) thay vì hy sinh accuracy.

## 🧠 Kiểm tra nhanh

1. Vì sao `CountVectorizer` phải `fit` trên tập train chứ không phải toàn bộ dữ liệu?
2. Multinomial NB là model tuyến tính theo vector đếm — trọng số của nó là gì?
3. Bernoulli NB khác Multinomial NB ở cách xử lý từ **không xuất hiện** thế nào?
4. Đặt $\alpha = 0$ thì chuyện gì xảy ra với một từ chỉ xuất hiện trong review tiêu cực ở tập train?
5. Vì sao tăng $\alpha$ lại làm log-loss tốt hơn trong thí nghiệm ở mục 6?

## 📌 Tóm tắt

- Văn bản → bag-of-words (thưa); Multinomial NB = đếm + Laplace smoothing; nhanh, baseline mạnh cho văn bản.
- Multinomial NB là model tuyến tính với trọng số $\log\theta_{cw}$ — diễn giải dễ.
- Bag-of-words mất thứ tự từ → yếu với phủ định; n-gram giúp một phần.
- Bernoulli kém khi rất ít dữ liệu, Complement giúp lớp nhỏ; chọn $\alpha$ bằng CV theo đúng metric cần tối ưu.